In [1]:
# Load the original dataset and build customer features
import pandas as pd
import numpy as np

df = pd.read_csv("Nassau Candy Distributor.csv")

# Aggregate transaction records into customer profiles
customer_features = (
    df.groupby("Customer ID")
    .agg(
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Gross Profit", "sum"),
        Total_Units=("Units", "sum"),
        Order_Count=("Order ID", "nunique"),
        Transaction_Count=("Row ID", "count"),
        Product_Count=("Product ID", "nunique")
    )
    .reset_index()
)

# Calculate additional customer behavior features
customer_features["Average_Sales_Per_Order"] = (
    customer_features["Total_Sales"]
    / customer_features["Order_Count"].clip(lower=1)
)

customer_features["Average_Units_Per_Order"] = (
    customer_features["Total_Units"]
    / customer_features["Order_Count"].clip(lower=1)
)

customer_features["Gross_Profit_Margin"] = (
    customer_features["Total_Profit"]
    / customer_features["Total_Sales"].replace(0, np.nan)
    * 100
)

customer_features.replace(
    [np.inf, -np.inf], np.nan, inplace=True
)

print("Customer feature shape:", customer_features.shape)
display(customer_features.head())

Customer feature shape: (5044, 10)


,Customer ID,Total_Sales,Total_Profit,Total_Units,Order_Count,Transaction_Count,Product_Count,Average_Sales_Per_Order,Average_Units_Per_Order,Gross_Profit_Margin
0,100006,11.25,7.35,3,1,1,1,11.250000,3.0,65.333333
1,100013,18.00,12.50,5,1,3,1,18.000000,5.0,69.444444
2,100041,21.25,14.31,6,3,3,3,7.083333,2.0,67.341176
3,100048,36.80,24.08,11,2,3,2,18.400000,5.5,65.434783
4,100055,18.30,12.40,5,2,2,2,9.150000,2.5,67.759563


In [2]:
# Add categorical customer preferences
def get_mode(series):
    modes = series.mode()
    return modes.iloc[0] if not modes.empty else "Unknown"

preferences = (
    df.groupby("Customer ID")
    .agg(
        Preferred_Division=("Division", get_mode),
        Main_Region=("Region", get_mode),
        Preferred_Ship_Mode=("Ship Mode", get_mode)
    )
    .reset_index()
)

customer_features = customer_features.merge(
    preferences,
    on="Customer ID",
    how="left",
    validate="one_to_one"
)

display(customer_features.head())

,Customer ID,Total_Sales,Total_Profit,Total_Units,Order_Count,Transaction_Count,Product_Count,Average_Sales_Per_Order,Average_Units_Per_Order,Gross_Profit_Margin,Preferred_Division,Main_Region,Preferred_Ship_Mode
0,100006,11.25,7.35,3,1,1,1,11.250000,3.0,65.333333,Chocolate,Atlantic,Standard Class
1,100013,18.00,12.50,5,1,3,1,18.000000,5.0,69.444444,Chocolate,Pacific,Standard Class
2,100041,21.25,14.31,6,3,3,3,7.083333,2.0,67.341176,Chocolate,Interior,Standard Class
3,100048,36.80,24.08,11,2,3,2,18.400000,5.5,65.434783,Chocolate,Atlantic,Standard Class
4,100055,18.30,12.40,5,2,2,2,9.150000,2.5,67.759563,Chocolate,Atlantic,Standard Class


In [3]:
# Check feature quality
feature_columns = [
    "Total_Sales",
    "Total_Profit",
    "Total_Units",
    "Order_Count",
    "Transaction_Count",
    "Product_Count",
    "Average_Sales_Per_Order",
    "Average_Units_Per_Order",
    "Gross_Profit_Margin"
]

print("Missing values:")
display(customer_features[feature_columns].isnull().sum())

print("\nFeature statistics:")
display(customer_features[feature_columns].describe().round(2))

print("\nDuplicate customer IDs:",
      customer_features["Customer ID"].duplicated().sum())

Missing values:


,0
Total_Sales,0
Total_Profit,0
Total_Units,0
Order_Count,0
Transaction_Count,0
Product_Count,0
Average_Sales_Per_Order,0
Average_Units_Per_Order,0
Gross_Profit_Margin,0



Feature statistics:


,Total_Sales,Total_Profit,Total_Units,Order_Count,Transaction_Count,Product_Count,Average_Sales_Per_Order,Average_Units_Per_Order,Gross_Profit_Margin
count,5044.00,5044.00,5044.00,5044.00,5044.00,5044.00,5044.00,5044.00,5044.00
mean,28.11,18.53,7.66,1.69,2.02,1.67,15.93,4.38,66.44
std,27.41,17.27,6.62,1.00,1.51,0.91,10.77,2.41,6.01
min,1.25,0.25,1.00,1.00,1.00,1.00,1.25,1.00,7.69
25%,10.80,7.35,3.00,1.00,1.00,1.00,9.34,2.75,65.33
50%,20.90,13.53,6.00,1.00,1.00,1.00,13.96,4.00,66.67
75%,35.75,24.08,10.00,2.00,3.00,2.00,19.69,5.50,69.44
max,454.40,256.60,110.00,15.00,25.00,6.00,200.00,19.00,78.20



Duplicate customer IDs: 0


In [4]:
# Select numerical features
numeric_features = [
    "Total_Sales",
    "Total_Profit",
    "Total_Units",
    "Order_Count",
    "Product_Count",
    "Average_Sales_Per_Order",
    "Average_Units_Per_Order"
]

X_numeric = customer_features[numeric_features].copy()

print("Selected features:", numeric_features)
print("Input shape:", X_numeric.shape)

display(X_numeric.head())

Selected features: ['Total_Sales', 'Total_Profit', 'Total_Units', 'Order_Count', 'Product_Count', 'Average_Sales_Per_Order', 'Average_Units_Per_Order']
Input shape: (5044, 7)


,Total_Sales,Total_Profit,Total_Units,Order_Count,Product_Count,Average_Sales_Per_Order,Average_Units_Per_Order
0,11.25,7.35,3,1,1,11.250000,3.0
1,18.00,12.50,5,1,1,18.000000,5.0
2,21.25,14.31,6,3,3,7.083333,2.0
3,36.80,24.08,11,2,2,18.400000,5.5
4,18.30,12.40,5,2,2,9.150000,2.5


In [5]:
# Scale the numerical features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_numeric)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=numeric_features,
    index=customer_features.index
)

print("Scaled data shape:", X_scaled.shape)
display(X_scaled.head().round(3))

print("\nScaled feature means:")
display(X_scaled.mean().round(3))

print("\nScaled feature standard deviations:")
display(X_scaled.std(ddof=0).round(3))

Scaled data shape: (5044, 7)


,Total_Sales,Total_Profit,Total_Units,Order_Count,Product_Count,Average_Sales_Per_Order,Average_Units_Per_Order
0,-0.615,-0.647,-0.705,-0.695,-0.736,-0.435,-0.572
1,-0.369,-0.349,-0.402,-0.695,-0.736,0.192,0.258
2,-0.250,-0.244,-0.251,1.305,1.456,-0.822,-0.987
3,0.317,0.322,0.504,0.305,0.360,0.229,0.465
4,-0.358,-0.355,-0.402,0.305,0.360,-0.630,-0.779



Scaled feature means:


,0
Total_Sales,-0.0
Total_Profit,0.0
Total_Units,-0.0
Order_Count,0.0
Product_Count,-0.0
Average_Sales_Per_Order,0.0
Average_Units_Per_Order,-0.0



Scaled feature standard deviations:


,0
Total_Sales,1.0
Total_Profit,1.0
Total_Units,1.0
Order_Count,1.0
Product_Count,1.0
Average_Sales_Per_Order,1.0
Average_Units_Per_Order,1.0


In [6]:
# Save the prepared datasets
customer_features.to_csv(
    "Customer_Feature_Engineered.csv",
    index=False
)

X_scaled.to_csv(
    "Customer_Features_Scaled.csv",
    index=False
)

print("Feature engineering completed.")
print("Saved: Customer_Feature_Engineered.csv")
print("Saved: Customer_Features_Scaled.csv")

Feature engineering completed.
Saved: Customer_Feature_Engineered.csv
Saved: Customer_Features_Scaled.csv
